## Improving Search Results

### 1.Multi-Query Retrieval

In [21]:
from openai import OpenAI
from pydantic import BaseModel
import os

In [22]:
client = OpenAI()

In [23]:
question = "What are the benefits of renewable energy?"

query_prompt = f"""You are an AI language model assistant. Your task is
to create three alternative versions of the provided user query to
enhance the retrieval of relevant documents from a vector database.
By offering diverse variations of the query, your goal is to help
mitigate the limitations of distance-based similarity search. Provide
these alternative queries, each on a new line.

Original query: {question}"""

In [30]:
# send the query prompt to OpenAI
class QueryVariations(BaseModel):
    queries: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": query_prompt,
        },
    ],
    response_format=QueryVariations,
)

queries = completion.choices[0].message.parsed.queries

In [31]:
queries

['What environmental, economic, and public-health benefits result from using renewable energy sources?',
 'How does adopting wind, solar, hydro, and other renewables reduce greenhouse gas emissions, lower energy costs, and improve energy security?',
 'What are the long-term financial, job-creation, and grid-resilience advantages of shifting from fossil fuels to renewable energy?']

### 2.Query Routing System

In [37]:
from pydantic import BaseModel,Field
from openai import OpenAI
from typing import Literal

In [33]:
client = OpenAI()

In [34]:
user_queries = [
    {
        "query": "Who is the all-time top scorer in the FIFA World Cup?",
        "selected_data_source": None,
    },
    {
        "query": "What are the four Grand Slam tennis tournaments?",
        "selected_data_source": None,
    },
    {
        "query": "Did Manchester United win their last game?",
        "selected_data_source": None,
    },
]

In [35]:
prompt = f"""
You are an expert at routing a user question to the appropriate
data source. Given a user question choose which of the data sources
in list_of_data_sources is the best to answer the question.
"""

In [39]:
class RouterDecision(BaseModel):
    data_source: Literal[
        "general_football_knowledge",
        "general_tennis_knowledge",
        "latest_football_results_sql",
    ] = Field(
        ...,
        description="The best data source to answer the question."
    )

for user_query in user_queries:
    completion = client.beta.chat.completions.parse(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_query["query"]},
        ],
        model="gpt-5-mini",
        response_format=RouterDecision,
    )
    user_query["selected_data_source"] = completion.choices[0].message.parsed.data_source

In [41]:
user_queries

[{'query': 'Who is the all-time top scorer in the FIFA World Cup?',
  'selected_data_source': 'general_football_knowledge'},
 {'query': 'What are the four Grand Slam tennis tournaments?',
  'selected_data_source': 'general_tennis_knowledge'},
 {'query': 'Did Manchester United win their last game?',
  'selected_data_source': 'latest_football_results_sql'}]

### 3.Enhancing Retrieval Accuracy with HyDE

In [44]:
user_query = "What is the revenue of Company X in 2024?"

In [46]:
class HypotheticalDocuments(BaseModel):
    documents : list[str]

In [47]:
prompt = f"""
You are an AI assistant. Based on the user query below, generate
three hypothetical text chunks that contain relevant information to
answer the query.
"""

In [48]:
completion = client.beta.chat.completions.parse(
    messages=[
        {'role':'system','content':prompt},
        {'role':'user','content':user_query},
    ],
    model="gpt-5-mini",
    response_format=HypotheticalDocuments
)

In [49]:
hypothetical_documents = completion.choices[0].message.parsed.documents

In [51]:
len(hypothetical_documents)

3

In [57]:
for i in hypothetical_documents:
    print(i)

Hypothetical annual report excerpt (Company X, FY2024): Consolidated revenue for the year ended December 31, 2024 was $3,200,000,000, compared with $2,860,000,000 in 2023. (Note: excerpt is fictional and provided as a sample.)
Hypothetical press release (January 22, 2025): Company X reports full-year 2024 revenue of $3.18 billion, an 11.6% increase versus 2023; the company attributes the growth to expansion in cloud services and subscription offerings. (Sample text.)
Hypothetical analyst note (March 10, 2025): After adjustments to deferred revenue and one-time items, consensus estimate for Company X's 2024 revenue is $3.19 billion; management's earlier guidance ranged from $3.15–$3.25 billion. (This is a fictional analyst summary.)


### 4. Decomposing Complex Queries into Multiple Sub-Queries

In [58]:
from pydantic import BaseModel
from typing import Optional
from openai import OpenAI

In [59]:
class Question(BaseModel):
    question:str
    answer:Optional[str] = None

class Questions(BaseModel):
    questions:list[Question]

splitter_prompt = """
You are a helpful assistant for a RAG chatbot.

Your job is to break down complex questions into simpler ones that
are easy to answer. When the answers to these simpler questions are
combined, they should fully answer the original question. If the
question is already simple, leave it as it is. Handle one question
at a time.

Example:
    1. Query: Did Microsoft or Google make more money last year?

Decomposed Questions:
    1. How much profit did Microsoft make last year?
    2. How much profit did Google make last year?
"""

In [60]:
query = (
    "What are the benefits of renewable energy compared to "
    "fossil fuels?"
)

In [61]:
completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {'role':'system','content':splitter_prompt},
        {'role':'user','content':query}
    ],
    response_format=Questions
)

In [62]:
decomposed_questions = completion.choices[0].message.parsed.questions

In [64]:
len(decomposed_questions)

9

### 5. Improving Retrieval Relevancy with Reranking Methods

In [65]:
import textwrap

In [66]:
text_chunks = {
    1: "Tesla's Supercharger network and tech lead face rising "
    "competition from BYD and established automakers.",
    2: "Tesla's production grows, but price competition threatens "
    "its market share.",
    3: "The automotive industry is shifting to EVs due to climate "
    "change and regulations.",
    4: "Semiconductor shortages are disrupting automotive supply "
    "chains.",
    5: "Consumer demand for autonomous driving and advanced tech "
    "impacts EV competition.",
}

In [67]:
prompt = textwrap.dedent(
    f""""
    Query: Will Tesla remain the market leader in electric vehicles?

    Documents:
        1. {text_chunks[1]}
        2. {text_chunks[2]}
        3. {text_chunks[3]}
        4. {text_chunks[4]}
        5. {text_chunks[5]}

    Instructions:
    Please assess the relevance of each document to the query and
    provide a relevance score from 1 to 5, where 5 is the most relevant.

    Relevance Scores:
    """
)